In [4]:
!pip install ultralytics
from ultralytics import YOLO
import os
import yaml
import glob

# 1. Find the data.yaml file automatically
dataset_yaml_path = ""
for root, dirs, files in os.walk("/kaggle/input"):
    if "data.yaml" in files:
        dataset_yaml_path = os.path.join(root, "data.yaml")
        break

if not dataset_yaml_path:
    print("❌ ERROR: Could not find data.yaml. Are you sure you added the dataset?")
else:
    print(f"✅ FOUND YAML AT: {dataset_yaml_path}")
    base_dir = os.path.dirname(dataset_yaml_path)

    # 2. HELPER: Find exactly where the images are
    def find_image_folder(start_path, search_name):
        # First, try to find a folder matching the name (e.g., "valid", "val", "test")
        candidate_dir = ""
        for root, dirs, files in os.walk(start_path):
            if os.path.basename(root).lower() in [s.lower() for s in search_name]:
                candidate_dir = root
                break
        
        if not candidate_dir:
            return None
            
        # Now, check if images are DIRECTLY here or in a subfolder
        # Check for images in candidate_dir
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in os.listdir(candidate_dir)):
            return candidate_dir
            
        # Check for 'images' subfolder
        img_sub = os.path.join(candidate_dir, "images")
        if os.path.exists(img_sub):
            return img_sub
            
        return candidate_dir # Fallback

    # 3. Auto-Detect Real Paths
    real_train_path = find_image_folder(base_dir, ["train", "training"])
    real_val_path = find_image_folder(base_dir, ["valid", "val", "validation", "test"]) # Use test as fallback for val
    
    print(f"🕵️ DETECTED PATHS:\n  Train: {real_train_path}\n  Val:   {real_val_path}")

    if not real_train_path or not real_val_path:
        print("❌ CRITICAL: Could not find image folders automatically.")
        # Print directory structure to help debug
        print(f"\n📂 Listing {base_dir}:")
        for root, dirs, files in os.walk(base_dir):
            level = root.replace(base_dir, '').count(os.sep)
            indent = ' ' * 4 * (level)
            print(f"{indent}{os.path.basename(root)}/")
            if level < 2: # Only show files for top levels
                for f in files[:3]: print(f"{indent}    {f}")
                if len(files) > 3: print(f"{indent}    ...")
    else:
        # 4. Create Fixed Config
        with open(dataset_yaml_path, 'r') as f:
            data = yaml.safe_load(f)

        data['train'] = real_train_path
        data['val'] = real_val_path
        data['test'] = real_val_path # Optional

        fixed_yaml_path = "/kaggle/working/data_fixed.yaml"
        with open(fixed_yaml_path, "w") as f:
            yaml.dump(data, f)

        print("✅ CONFIG FIXED & SAVED!")

        # 5. TRAIN
        model = YOLO('yolov8n.pt')
        results = model.train(
            data=fixed_yaml_path,
            epochs=50,
            imgsz=640,
            batch=32,
            device=0,
            name='waste_yolo_kaggle_final'
        )

✅ FOUND YAML AT: /kaggle/input/garbage-data/data.yaml
🕵️ DETECTED PATHS:
  Train: /kaggle/input/garbage-data/paper/paper-1/train/images
  Val:   /kaggle/input/garbage-data/paper/paper-1/valid/images
✅ CONFIG FIXED & SAVED!
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, 